## 1. Imports

In [12]:
import os
import time
import json
import requests
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import FunctionTool

## 2. Define helper functions

In [ ]:
# Function 1: Weather API
def fetch_weather(location: str) -> str:
    """Fetches the weather information for the specified location using WeatherAPI.com.
    
    :param location: The location to fetch weather for.
    :return: Weather information as a JSON string.
    """
    api_key = "your-key"
    base_url = "http://api.weatherapi.com/v1/current.json"
    
    try:
        params = {
            'key': api_key,
            'q': location,
            'aqi': 'no'
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        temperature = data['current']['temp_c']
        weather_condition = data['current']['condition']['text']
        humidity = data['current']['humidity']
        wind_speed = data['current']['wind_kph']
        
        weather_info = f"{weather_condition}, {temperature}°C, Humidity: {humidity}%, Wind: {wind_speed} km/h"
        
        return json.dumps({
            "weather": weather_info,
            "location": location,
            "temperature": temperature,
            "condition": weather_condition,
            "humidity": humidity,
            "wind_speed": wind_speed
        })
        
    except Exception as e:
        return json.dumps({
            "error": f"Error fetching weather data: {str(e)}",
            "location": location
        })

In [ ]:
# Function 2: Currency Exchange Rate API
def get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    """Gets the exchange rate between two currencies and converts an amount.
    
    :param from_currency: The currency code to convert from (e.g., USD, EUR, GBP).
    :param to_currency: The currency code to convert to (e.g., USD, EUR, GBP).
    :param amount: The amount to convert (default: 1.0).
    :return: Exchange rate information as a JSON string.
    """
    base_url = f"https://api.exchangerate-api.com/v4/latest/{from_currency.upper()}"
    
    try:
        response = requests.get(base_url)
        response.raise_for_status()
        data = response.json()
        
        if to_currency.upper() not in data['rates']:
            return json.dumps({
                "error": f"Currency {to_currency} not found",
                "from_currency": from_currency,
                "to_currency": to_currency
            })
        
        rate = data['rates'][to_currency.upper()]
        converted_amount = amount * rate
        
        return json.dumps({
            "from_currency": from_currency.upper(),
            "to_currency": to_currency.upper(),
            "exchange_rate": rate,
            "original_amount": amount,
            "converted_amount": round(converted_amount, 2),
            "result": f"{amount} {from_currency.upper()} = {round(converted_amount, 2)} {to_currency.upper()}"
        })
        
    except Exception as e:
        return json.dumps({
            "error": f"Error fetching exchange rate: {str(e)}",
            "from_currency": from_currency,
            "to_currency": to_currency
        })

In [ ]:
# Function 3: Math Calculator (Local Processing)
def calculate_math(expression: str) -> str:
    """Evaluates a mathematical expression safely.
    
    :param expression: The mathematical expression to evaluate (e.g., "2 + 2", "10 * 5 + 3").
    :return: Calculation result as a JSON string.
    """
    try:
        # Use a safe evaluation method - restrict to basic math operations
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            return json.dumps({
                "error": "Invalid characters in expression. Only numbers and basic operators (+, -, *, /, parentheses) are allowed.",
                "expression": expression
            })
        
        # Evaluate the expression
        result = eval(expression, {"__builtins__": {}}, {})
        
        return json.dumps({
            "expression": expression,
            "result": result,
            "formatted": f"{expression} = {result}"
        })
        
    except ZeroDivisionError:
        return json.dumps({
            "error": "Division by zero",
            "expression": expression
        })
    except Exception as e:
        return json.dumps({
            "error": f"Error calculating expression: {str(e)}",
            "expression": expression
        })


## 3. Initialize project client

In [ ]:
# Load configuration
def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None


In [ ]:
# Initialize project client
try:
    start_dir = os.getcwd()
    file_path = find_cred_json(start_dir)
    
    if not file_path:
        raise FileNotFoundError("Could not find cred.json file")
    
    print(f"Found cred.json at: {file_path}")
    
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)
    
    project_client = AIProjectClient(
        credential=DefaultAzureCredential(),
        endpoint=loaded_config["PROJECT_ENDPOINT"],
    )
    
    model_name = loaded_config.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")
    print("✅ Successfully initialized AIProjectClient")
    
except Exception as e:
    print(f"❌ Error initializing project client: {e}")
    exit(1)

## 4.Create agent

In [ ]:
# Define user functions - now with three functions!
user_functions = {fetch_weather, get_exchange_rate, calculate_math}

# Initialize the FunctionTool with all user-defined functions
functions = FunctionTool(functions=user_functions)

In [ ]:
def create_multi_function_agent():
    """Create an agent with multiple function capabilities."""
    try:
        agent = project_client.agents.create_agent(
            model=model_name,
            name="multi-function-agent",
            instructions="""You are a helpful assistant with multiple capabilities:
            1. Weather: Use fetch_weather to get weather information for any location
            2. Currency: Use get_exchange_rate to convert between currencies
            3. Math: Use calculate_math to evaluate mathematical expressions
            
            Always use the appropriate function based on the user's request.""",
            tools=functions.definitions,
        )
        print(f"✅ Created agent, ID: {agent.id}")
        return agent
    except Exception as e:
        print(f"❌ Error creating agent: {e}")
        return None

## 5.Create thread

In [ ]:
def run_conversation(agent, user_message):
    """Run a conversation with the multi-function agent."""
    try:
        # Create a thread for communication
        thread = project_client.agents.threads.create()
        print(f"📝 Created thread, ID: {thread.id}")

        # Send a message to the thread
        message = project_client.agents.messages.create(
            thread_id=thread.id,
            role="user",
            content=user_message,
        )
        print(f"➕ Created message, ID: {message.id}")

        # Create and process a run
        run = project_client.agents.runs.create(
            thread_id=thread.id, 
            agent_id=agent.id
        )
        print(f"▶️ Created run, ID: {run.id}")

        # Poll the run status
        max_attempts = 60
        attempts = 0
        
        while run.status in ["queued", "in_progress", "requires_action"] and attempts < max_attempts:
            time.sleep(1)
            attempts += 1
            run = project_client.agents.runs.get(thread_id=thread.id, run_id=run.id)
            
            if run.status == "requires_action":
                print("🔧 Processing function calls...")
                tool_calls = run.required_action.submit_tool_outputs.tool_calls
                tool_outputs = []
                
                for tool_call in tool_calls:
                    function_name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)
                    
                    print(f"   📞 Calling function: {function_name}")
                    print(f"      Arguments: {args}")
                    
                    # Route to appropriate function
                    if function_name == "fetch_weather":
                        output = fetch_weather(args.get("location", "New York"))
                    elif function_name == "get_exchange_rate":
                        output = get_exchange_rate(
                            args.get("from_currency", "USD"),
                            args.get("to_currency", "EUR"),
                            args.get("amount", 1.0)
                        )
                    elif function_name == "calculate_math":
                        output = calculate_math(args.get("expression", "0"))
                    else:
                        output = json.dumps({"error": f"Unknown function: {function_name}"})
                    
                    tool_outputs.append({
                        "tool_call_id": tool_call.id, 
                        "output": output
                    })
                
                # Submit the tool outputs
                project_client.agents.runs.submit_tool_outputs(
                    thread_id=thread.id, 
                    run_id=run.id, 
                    tool_outputs=tool_outputs
                )
                print("   ✅ Submitted tool outputs")

        print(f"🤖 Run completed with status: {run.status}")

        # Fetch and display messages
        if run.status == "completed":
            messages = project_client.agents.messages.list(thread_id=thread.id)
            message_list = list(messages)
            
            print("\n💬 Conversation:")
            for message in message_list:
                role = message.role
                if message.content:
                    for content in message.content:
                        if hasattr(content, 'text') and content.text:
                            print(f"   {role.upper()}: {content.text.value}")
        
        return thread, run
        
    except Exception as e:
        print(f"❌ Error in conversation: {e}")
        import traceback
        traceback.print_exc()
        return None, None

## 6.Run agent and handle tool calls

In [ ]:
# Main execution
if __name__ == "__main__":
    # Create the multi-function agent
    agent = create_multi_function_agent()
    
    if agent:
        # Test cases for all three functions
        test_queries = [
            "What's the weather in Tokyo and Paris?",
            "Convert 100 USD to EUR and GBP",
            "Calculate (25 * 4) + (100 / 2) - 10"
        ]
        
        # You can also test with a combined query:
        # "What's the weather in New York? Also convert 50 USD to EUR. And calculate 15 * 8"
        
        for i, query in enumerate(test_queries, 1):
            print(f"\n{'='*60}")
            print(f"🗣️ Test Query {i}: {query}")
            print(f"{'='*60}")
            
            thread, run = run_conversation(agent, query)
            
            if i < len(test_queries):
                print("\n⏳ Waiting 2 seconds before next query...")
                time.sleep(2)
        
        # Cleanup
        try:
            project_client.agents.delete_agent(agent.id)
            print("\n🗑️ Deleted agent")
        except Exception as e:
            print(f"⚠️ Could not delete agent: {e}")